
## **How Swiggy Decides What's Selling and What's Not - Using Data**

**GMV (Gross Merchandise Value)**\
What it is: Total value of all orders placed\
GMV = sum(order_value)\
Use: Measures platform size and growth

👉 Think: “How much money flowed through the platform?”\

**AOV (Average Order Value)**\
What it is: Average spend per order\
AOV = GMV / number_of_orders\
Use: Indicates customer spending behavior

👉 High AOV = customers buying more per order\

**Whale Orders**\
What it is: Very high-value orders (top spenders)\
Top 5% (P95), or Orders above a high threshold

👉 Not a fixed number—depends on dataset

--

## Dataset: `swiggy_orders.csv`
| Column | Type | Description |
|---|---|---|
| order_id | int | Unique order identifier (has duplicates!) |
| restaurant_name | str | Name of restaurant |
| city | str | 12 Indian cities |
| category | str | Cuisine category |
| order_value | float | Order amount in INR |
| delivery_time_mins | int | Delivery duration (has nulls + negatives) |
| rating | float | Customer rating 1–5 (has nulls + values > 5) |
| order_date | datetime | Date and time of order |
| time_slot | str | breakfast / lunch / snacks / dinner |
| is_reorder | bool | Returning customer order |
| discount_applied | float | Discount amount (NaN = no discount) |
| platform | str | iOS / Android / Web |

# **1. NumPy Arrays & Vectorised Operations**

In [ ]:
import numpy as np
import pandas as pd
# Load just the order values for now
df_raw = pd.read_csv('swiggy_orders.csv')
order_values = df_raw['order_value'].values  # Convert to NumPy array

print(f"Data type: {type(order_values)}")
print(f"Shape: {order_values.shape}")
print(f"First 5 values: {order_values[:5]}")


Data type: <class 'numpy.ndarray'>
Shape: (2050,)
First 5 values: [745.02 114.67 243.78 274.41 343.46]


In [ ]:
# ── Basic Aggregations ────────────────────────────────────────────────
gmv = np.sum(order_values)
aov = np.mean(order_values)
median_order = np.median(order_values)
p90_order = np.percentile(order_values, 90)

print("=" * 40)
print("  SWIGGY REVENUE SNAPSHOT")
print("=" * 40)
print(f"  Total GMV (Gross Merch Value) : ₹{gmv:,.0f}")
print(f"  Average Order Value (AOV)     : ₹{aov:.2f}")
print(f"  Median Order Value            : ₹{median_order:.2f}")
print(f"  90th Percentile Order         : ₹{p90_order:.2f}")
print("=" * 40)

  SWIGGY REVENUE SNAPSHOT
  Total GMV (Gross Merch Value) : ₹910,878
  Average Order Value (AOV)     : ₹444.33
  Median Order Value            : ₹346.12
  90th Percentile Order         : ₹706.26


In [ ]:
# ── Revenue Tier Segmentation using Boolean Masks ─────────────────────
# Business context: Swiggy categorises orders by value to manage delivery priority

micro   = order_values < 200
mid     = (order_values >= 200) & (order_values < 500)
premium = (order_values >= 500) & (order_values < 1000)
whale   = order_values >= 1000

print("Order Value Tier Breakdown")
print("-" * 45)
for label, mask in [("Micro (<₹200)", micro), ("Mid (₹200-500)", mid),
                     ("Premium (₹500-1000)", premium), ("Whale (>₹1000)", whale)]:
    count = mask.sum()
    revenue = order_values[mask].sum()
    share = revenue / gmv * 100
    print(f"  {label:<22} {count:>5} orders | ₹{revenue:>10,.0f} | {share:>5.1f}% of GMV")

Order Value Tier Breakdown
---------------------------------------------
  Micro (<₹200)            281 orders | ₹    43,493 |   4.8% of GMV
  Mid (₹200-500)          1235 orders | ₹   405,477 |  44.5% of GMV
  Premium (₹500-1000)      503 orders | ₹   338,185 |  37.1% of GMV
  Whale (>₹1000)            31 orders | ₹   123,722 |  13.6% of GMV


# **2. Load & Inspect Real Datasets**

In [ ]:
df = pd.read_csv('swiggy_orders.csv')

In [ ]:
print(df.dtypes)

order_id                int64
restaurant_name        object
city                   object
category               object
order_value           float64
delivery_time_mins    float64
rating                float64
order_date             object
time_slot              object
is_reorder               bool
discount_applied      float64
platform               object
dtype: object


In [ ]:
df = pd.read_csv('swiggy_orders.csv', parse_dates=['order_date'])
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")

Shape: 2,050 rows × 12 columns


In [ ]:
print(df.dtypes)

order_id                       int64
restaurant_name               object
city                          object
category                      object
order_value                  float64
delivery_time_mins           float64
rating                       float64
order_date            datetime64[ns]
time_slot                     object
is_reorder                      bool
discount_applied             float64
platform                      object
dtype: object


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2050 entries, 0 to 2049
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   order_id            2050 non-null   int64         
 1   restaurant_name     2050 non-null   object        
 2   city                2050 non-null   object        
 3   category            2050 non-null   object        
 4   order_value         2050 non-null   float64       
 5   delivery_time_mins  1884 non-null   float64       
 6   rating              1846 non-null   float64       
 7   order_date          2050 non-null   datetime64[ns]
 8   time_slot           1989 non-null   object        
 9   is_reorder          2050 non-null   bool          
 10  discount_applied    1133 non-null   float64       
 11  platform            2050 non-null   object        
dtypes: bool(1), datetime64[ns](1), float64(4), int64(1), object(5)
memory usage: 178.3+ KB


In [ ]:
df.describe()

,order_id,order_value,delivery_time_mins,rating,order_date,discount_applied
count,2050.000000,2050.000000,1884.000000,1846.00000,2050,1133.000000
mean,101003.745366,444.330673,41.559979,3.99507,2024-11-15 23:00:29.590244096,65.402445
min,100001.000000,80.000000,-28.000000,2.30000,2024-10-01 09:21:00,10.080000
25%,100503.250000,243.802500,34.000000,3.60000,2024-10-23 20:46:15,38.980000
50%,101003.500000,346.125000,42.000000,4.00000,2024-11-15 20:47:00,65.300000
75%,101505.750000,509.952500,50.000000,4.40000,2024-12-08 13:27:45,92.470000
max,102000.000000,5705.620000,79.000000,6.40000,2024-12-29 22:28:00,119.880000
std,578.339815,497.157495,12.335249,0.58240,NaN,31.501201


In [ ]:
df.head()

,order_id,restaurant_name,city,category,order_value,delivery_time_mins,rating,order_date,time_slot,is_reorder,discount_applied,platform
0,101569,Sweet Truth,Hyderabad,Desserts,745.02,46.0,NaN,2024-12-18 19:51:00,dinner,False,97.73,iOS
1,101963,Dum Pukht,Kolkata,Biryani,114.67,69.0,3.9,2024-10-26 13:16:00,lunch,False,NaN,Android
2,101693,Sweet Truth,Delhi,Desserts,243.78,18.0,3.3,2024-10-01 21:54:00,dinner,False,58.94,Android
3,101873,Behrouz Biryani,Delhi,Biryani,274.41,35.0,3.6,2024-10-30 18:50:00,snacks,False,101.67,Web
4,100263,Raw Pressery Kitchen,Bangalore,Healthy,343.46,32.0,NaN,2024-12-22 12:07:00,lunch,False,13.51,Android


In [ ]:
print("Unique value counts:")
print(df.nunique().sort_values())

Unique value counts:
is_reorder               2
platform                 3
time_slot                4
category                12
city                    12
rating                  36
restaurant_name         60
delivery_time_mins      69
discount_applied      1046
order_value           1881
order_date            1976
order_id              2000
dtype: int64


In [ ]:
# ── Spot Anomalies ────────────────────────────────────────────────────
# ⚠️ Ratings above 5.0 — impossible, data error
bad_ratings = df[pd.to_numeric(df['rating'], errors='coerce') > 5.0]
print(f"Orders with rating > 5.0: {len(bad_ratings)}")
display(bad_ratings[['order_id', 'restaurant_name', 'rating']].head())

# ⚠️ Negative delivery times — physically impossible
bad_delivery = df[pd.to_numeric(df['delivery_time_mins'], errors='coerce') < 0]
print(f"\nOrders with negative delivery time: {len(bad_delivery)}")
display(bad_delivery[['order_id', 'city', 'delivery_time_mins']].head())


Orders with rating > 5.0: 8


,order_id,restaurant_name,rating
213,100594,Wok To Walk,5.4
454,101229,Behrouz Biryani,6.0
485,101745,Paradise Biryani,5.5
580,100417,MTR,6.1
667,100632,Wat-A-Burger,5.1



Orders with negative delivery time: 6


,order_id,city,delivery_time_mins
335,101516,Bangalore,-26.0
1241,101495,Hyderabad,-28.0
1414,101367,Chennai,-2.0
1747,100560,Hyderabad,-12.0
1990,100401,Bangalore,-28.0


In [ ]:
# ── Order date range ──────────────────────────────────────────────────
print(f"Date range: {df['order_date'].min()} → {df['order_date'].max()}")
print(f"Total days covered: {(df['order_date'].max() - df['order_date'].min()).days} days")
print(f"\nOrders by platform:")
print(df['platform'].value_counts())

Date range: 2024-10-01 09:21:00 → 2024-12-29 22:28:00
Total days covered: 89 days

Orders by platform:
platform
Android    1115
iOS         628
Web         307
Name: count, dtype: int64


# **3. Filter, Sort & Slice**

Which restaurants are top performers in high value cities?\
High value orders from bangalore in the dinner slot

In [ ]:
# ── Boolean mask: Bangalore orders above ₹400 ────────────────────────
blr_high=df[(df['city']=='Bangalore')&(df['order_value']>400)]
print(f"Bangalore orders above ₹400: {len(blr_high):,}")
print(f"Combined GMV: ₹{blr_high['order_value'].sum():,.0f}")
display(blr_high[['restaurant_name', 'category', 'order_value', 'time_slot']].head())

Bangalore orders above ₹400: 169
Combined GMV: ₹114,418


,restaurant_name,category,order_value,time_slot
26,Green Bowl,Healthy,861.24,dinner
29,Wok To Walk,Chinese,621.54,dinner
41,Udupi Palace,South Indian,680.65,lunch
44,Chai Point,Beverages,488.42,snacks
47,Udupi Palace,South Indian,629.30,dinner


In [ ]:
# Get rows 0-4, specific columns by NAME → .loc
print("Using .loc (label-based):")
display(df.loc[0:4, ['order_id', 'city', 'order_value']])

# Get first 3 rows, first 4 columns by POSITION → .iloc
print("\nUsing .iloc (position-based):")
display(df.iloc[:3, :4])

Using .loc (label-based):


,order_id,city,order_value
0,101569,Hyderabad,745.02
1,101963,Kolkata,114.67
2,101693,Delhi,243.78
3,101873,Delhi,274.41
4,100263,Bangalore,343.46



Using .iloc (position-based):


,order_id,restaurant_name,city,category
0,101569,Sweet Truth,Hyderabad,Desserts
1,101963,Dum Pukht,Kolkata,Biryani
2,101693,Sweet Truth,Delhi,Desserts


Top 20 restaurants by total revenue

In [ ]:
# ── Top 20 restaurants by total revenue ───────────────────────────────
top_restaurants = (
    df.groupby('restaurant_name')['order_value']
    .sum()
    .sort_values(ascending=False)
    .head(20)
    .reset_index()
    .rename(columns={'order_value': 'total_revenue'})
)
top_restaurants['total_revenue'] = top_restaurants['total_revenue'].map('₹{:,.0f}'.format)
print("Top 20 Restaurants by Revenue:")
display(top_restaurants)

Top 20 Restaurants by Revenue:


,restaurant_name,total_revenue
0,Dum Pukht,"₹43,076"
1,Behrouz Biryani,"₹37,902"
2,Empire Restaurant,"₹35,974"
3,Saffron,"₹32,548"
4,Punjab Grill,"₹31,096"
5,Paradise Biryani,"₹30,789"
6,Barbeque Nation,"₹28,843"
7,Moti Mahal,"₹26,935"
8,Bawarchi,"₹25,864"
9,Murugan Idli Shop,"₹25,667"


In [ ]:
# ── Multi-condition filter: Loyal Dinner Crowd ────────────────────────
# Business question: Who are our most loyal, high-value dinner customers?
loyal_dinner = df[
    (df['time_slot'] == 'dinner') &
    (pd.to_numeric(df['rating'], errors='coerce') >= 4.0) &
    (df['is_reorder'] == True) &
    (df['order_value'] >= 300)
]

print(f"Loyal dinner crowd size: {len(loyal_dinner):,} orders")
print(f"Avg order value: ₹{loyal_dinner['order_value'].mean():.2f}")
print(f"Top 5 cities for this segment:")
print(loyal_dinner['city'].value_counts().head())

Loyal dinner crowd size: 70 orders
Avg order value: ₹668.82
Top 5 cities for this segment:
city
Mumbai       13
Hyderabad    12
Delhi        11
Pune          9
Bangalore     8
Name: count, dtype: int64


In [ ]:
# ── Sort: Highest value orders per city ───────────────────────────────
top_per_city = (
    df.sort_values('order_value', ascending=False)
    .drop_duplicates(subset='city')
    [['city', 'restaurant_name', 'category', 'order_value', 'order_date']]
    .sort_values('city')
)
print("Single highest-value order per city:")
display(top_per_city)

Single highest-value order per city:


,city,restaurant_name,category,order_value,order_date
2042,Ahmedabad,Saffron,North Indian,3052.39,2024-10-23 08:06:00
1379,Bangalore,Burgerama,Burger,5275.75,2024-11-10 17:34:00
1630,Chennai,Dum Pukht,Biryani,5457.20,2024-11-10 20:39:00
1615,Delhi,Saravana Bhavan,South Indian,5705.62,2024-12-21 08:23:00
123,Hyderabad,La Pino'z,Pizza,4893.59,2024-11-10 18:25:00
122,Jaipur,La Pino'z,Pizza,4412.81,2024-12-23 22:58:00
733,Kochi,Bawarchi,Biryani,900.00,2024-10-20 14:03:00
488,Kolkata,Dum Pukht,Biryani,5129.80,2024-12-28 13:43:00
30,Lucknow,Pizza Hut,Pizza,724.73,2024-11-03 20:57:00
2045,Mumbai,Punjab Grill,North Indian,5643.41,2024-11-12 22:23:00


# **4. GroupBy & Aggregations**

City-wise GMV

In [ ]:
# ── City-wise GMV ─────────────────────────────────────────────────────
city_gmv = (
    df.groupby('city')['order_value']
    .agg(['sum', 'mean', 'count'])
    .rename(columns={'sum': 'GMV (₹)', 'mean': 'AOV (₹)', 'count': 'Orders'})
    .sort_values('GMV (₹)', ascending=False)
)
city_gmv['GMV Share %'] = (city_gmv['GMV (₹)'] / city_gmv['GMV (₹)'].sum() * 100).round(1)
city_gmv['GMV (₹)'] = city_gmv['GMV (₹)'].map('₹{:,.0f}'.format)
city_gmv['AOV (₹)'] = city_gmv['AOV (₹)'].map('₹{:.2f}'.format)
print("City-wise Revenue Breakdown:")
display(city_gmv)

City-wise Revenue Breakdown:


,GMV (₹),AOV (₹),Orders,GMV Share %
city,,,,
Bangalore,"₹185,393",₹423.27,438,20.4
Mumbai,"₹159,572",₹457.23,349,17.5
Delhi,"₹125,409",₹436.97,287,13.8
Hyderabad,"₹124,402",₹469.44,265,13.7
Chennai,"₹93,831",₹462.22,203,10.3
Kolkata,"₹67,550",₹482.50,140,7.4
Pune,"₹64,668",₹387.23,167,7.1
Ahmedabad,"₹30,802",₹427.81,72,3.4
Jaipur,"₹30,121",₹424.24,71,3.3


In [ ]:
# ── Category × Time Slot Revenue Matrix ───────────────────────────────
cat_slot = (
    df.dropna(subset=['time_slot'])
    .groupby(['category', 'time_slot'])
    .agg(
        total_revenue=('order_value', 'sum'),
        order_count=('order_id', 'count'),
        avg_order=('order_value', 'mean')
    )
    .round(2)
    .sort_values('total_revenue', ascending=False)
)
print("Category × Time Slot Performance (top 15):")
display(cat_slot.head(15))

Category × Time Slot Performance (top 15):


total_revenue  order_count  avg_order
category      time_slot                                       
Biryani       lunch           76406.25          128     596.92
North Indian  lunch           57154.23          121     472.35
Biryani       dinner          46447.43          111     418.45
South Indian  dinner          43633.44           95     459.30
              lunch           42173.35           89     473.86
North Indian  dinner          41107.52          103     399.10
Pizza         dinner          30624.91           69     443.84
Chinese       dinner          28849.42           74     389.86
Biryani       snacks          27874.12           71     392.59
Pizza         lunch           27403.95           64     428.19
North Indian  snacks          26238.37           53     495.06
Chinese       lunch           24633.45           62     397.31
Burger        dinner          24625.51           55     447.74
              lunch           23837.89           59     404.03
Rolls & Wraps lunch           23414.00           57     410.77

In [ ]:
# ── Pivot Table: Category vs City Revenue Matrix ──────────────────────
# This is the format a CFO or VP of Growth actually wants to see!
pivot = pd.pivot_table(
    df,
    values='order_value',
    index='category',
    columns='city',
    aggfunc='sum',
    fill_value=0
).round(0).astype(int)

# Show top 6 cities by total GMV
top_cities = df.groupby('city')['order_value'].sum().nlargest(6).index
print("Revenue Pivot: Category × Top 6 Cities (in ₹)")
display(pivot[top_cities].sort_values('Bangalore', ascending=False))

Revenue Pivot: Category × Top 6 Cities (in ₹)


city,Bangalore,Mumbai,Delhi,Hyderabad,Chennai,Kolkata
category,,,,,,
South Indian,30784,24916,18391,14939,8674,5560
Biryani,30729,25993,30909,16140,21761,20157
North Indian,24723,36357,10468,16806,16395,7365
Burger,22710,11141,14729,8996,4328,6032
Desserts,15672,7349,4662,7525,6895,6013
Pizza,13462,7863,10543,16202,6562,5282
Chinese,12245,16668,13968,10795,11211,5110
Rolls & Wraps,10773,10059,5063,8507,8711,2048
Healthy,9665,6434,4614,9113,3645,1145


In [ ]:
# ── % Contribution using .transform() ─────────────────────────────────
# transform() is powerful: it returns a series aligned with the original df
# Use case: "What % of each city's total GMV does each category represent?"

df['city_total_gmv'] = df.groupby('city')['order_value'].transform('sum')
df['category_share_in_city'] = (df['order_value'] / df['city_total_gmv'] * 100).round(2)

cat_city_share = (
    df.groupby(['city', 'category'])['category_share_in_city']
    .mean()
    .reset_index()
    .sort_values(['city', 'category_share_in_city'], ascending=[True, False])
    .groupby('city').head(3)  # Top 3 categories per city
)
print("Top 3 Revenue Categories per City:")
display(cat_city_share.head(30))

# Clean up temp columns
df.drop(columns=['city_total_gmv', 'category_share_in_city'], inplace=True)

Top 3 Revenue Categories per City:


,city,category,category_share_in_city
7,Ahmedabad,North Indian,2.172727
1,Ahmedabad,Biryani,1.339375
9,Ahmedabad,Rolls & Wraps,1.328000
13,Bangalore,Burger,0.299512
22,Bangalore,South Indian,0.272787
21,Bangalore,Seafood,0.231111
26,Chennai,Chinese,0.664444
24,Chennai,Biryani,0.643611
30,Chennai,North Indian,0.582000
36,Delhi,Biryani,0.432632


In [ ]:
# ── Reorder rate by time slot ─────────────────────────────────────────
reorder_by_slot = (
    df.dropna(subset=['time_slot'])
    .groupby('time_slot')['is_reorder']
    .agg(['sum', 'count'])
    .rename(columns={'sum': 'Reorders', 'count': 'Total'})
)
reorder_by_slot['Reorder Rate %'] = (reorder_by_slot['Reorders'] / reorder_by_slot['Total'] * 100).round(1)
print("Reorder Rate by Time Slot:")
display(reorder_by_slot)

Reorder Rate by Time Slot:


,Reorders,Total,Reorder Rate %
time_slot,,,
breakfast,74,197,37.6
dinner,258,681,37.9
lunch,280,711,39.4
snacks,161,400,40.2


# **5. Handle Nulls, Duplicates & Dirty Data**

In [ ]:
null_counts = df.isnull().sum()
null_pct = (null_counts / len(df) * 100).round(1)
null_audit = pd.DataFrame({'Null Count': null_counts, 'Null %': null_pct})
null_audit = null_audit[null_audit['Null Count'] > 0].sort_values('Null %', ascending=False)
print("Null Value Audit:")
display(null_audit)

Null Value Audit:


,Null Count,Null %
discount_applied,917,44.7
rating,204,10.0
delivery_time_mins,166,8.1
time_slot,61,3.0


In [ ]:
# ── Step 2: Strategy per column ───────────────────────────────────────
print(
    "Cleaning Strategy:\n"
    "--------------------------------------------------\n"
    "Column              | Null % | Strategy\n"
    "--------------------------------------------------\n"
    "discount_applied    | ~45%   | KEEP NaN = No discount applied (valid)\n"
    "rating              | ~10%   | FILL with median (avoid mean on skewed data)\n"
    "delivery_time_mins  | ~8%    | FILL with city-wise median (varies by city)\n"
    "time_slot           | ~3%    | FILL with mode (most common slot)\n"
    "--------------------------------------------------"
)

Cleaning Strategy:
--------------------------------------------------
Column              | Null % | Strategy
--------------------------------------------------
discount_applied    | ~45%   | KEEP NaN = No discount applied (valid)
rating              | ~10%   | FILL with median (avoid mean on skewed data)
delivery_time_mins  | ~8%    | FILL with city-wise median (varies by city)
time_slot           | ~3%    | FILL with mode (most common slot)
--------------------------------------------------


In [ ]:
df_clean = df.copy()

# Convert to numeric first (handles the bad string values)
df_clean['rating'] = pd.to_numeric(df_clean['rating'], errors='coerce')
df_clean['delivery_time_mins'] = pd.to_numeric(df_clean['delivery_time_mins'], errors='coerce')

# Fix ratings > 5.0 → set to NaN (impossible values)
df_clean.loc[df_clean['rating'] > 5.0, 'rating'] = np.nan

# Fix negative delivery times → set to NaN
df_clean.loc[df_clean['delivery_time_mins'] < 0, 'delivery_time_mins'] = np.nan

# Fill rating with global median
rating_median = df_clean['rating'].median()
df_clean['rating'] = df_clean['rating'].fillna(rating_median)
print(f"Rating: filled nulls with median = {rating_median}")

# Fill delivery_time_mins with city-wise median
df_clean['delivery_time_mins'] = df_clean.groupby('city')['delivery_time_mins'].transform(
    lambda x: x.fillna(x.median())
)
print(f"Delivery time: filled with city-wise medians")

# Fill time_slot with mode
mode_slot = df_clean['time_slot'].mode()[0]
df_clean['time_slot'] = df_clean['time_slot'].fillna(mode_slot)
print(f"Time slot: filled nulls with mode = '{mode_slot}'")

print(f"\nNulls remaining after cleaning:")
print(df_clean.isnull().sum()[df_clean.isnull().sum() > 0])

Rating: filled nulls with median = 4.0
Delivery time: filled with city-wise medians
Time slot: filled nulls with mode = 'lunch'

Nulls remaining after cleaning:
discount_applied    917
dtype: int64


In [ ]:
# ── Step 4: Remove Duplicates ─────────────────────────────────────────
print(f"Rows BEFORE deduplication: {len(df_clean):,}")

# Check duplicate order_ids
dupes = df_clean[df_clean.duplicated(subset='order_id', keep=False)]
print(f"Duplicate order_id rows found: {len(dupes)}")

# Keep the first occurrence
df_clean = df_clean.drop_duplicates(subset='order_id', keep='first').reset_index(drop=True)
print(f"Rows AFTER deduplication:  {len(df_clean):,}")
print(f"Rows removed: {len(df) - len(df_clean)}")

Rows BEFORE deduplication: 2,000
Duplicate order_id rows found: 0
Rows AFTER deduplication:  2,000
Rows removed: 50


In [ ]:
# ── Step 5: Outlier Detection (3σ rule) ───────────────────────────────
mean_ov = df_clean['order_value'].mean()
std_ov  = df_clean['order_value'].std()
lower   = mean_ov - 3 * std_ov
upper   = mean_ov + 3 * std_ov

outliers = df_clean[(df_clean['order_value'] < lower) | (df_clean['order_value'] > upper)]
print(f"Mean ± 3σ range: ₹{lower:.0f} to ₹{upper:.0f}")
print(f"Outliers detected: {len(outliers)}")
print(f"Outlier order values:")
display(outliers[['order_id', 'city', 'category', 'order_value', 'restaurant_name']].sort_values('order_value', ascending=False))

# Business decision: flag outliers but don't remove (they're real whale orders)
df_clean['is_outlier'] = (df_clean['order_value'] < lower) | (df_clean['order_value'] > upper)

Mean ± 3σ range: ₹-1049 to ₹1937
Outliers detected: 30
Outlier order values:


,order_id,city,category,order_value,restaurant_name
1583,100130,Delhi,South Indian,5705.62,Saravana Bhavan
1995,101604,Mumbai,North Indian,5643.41,Punjab Grill
1620,101223,Delhi,Biryani,5616.40,Paradise Biryani
1596,100720,Chennai,Biryani,5457.20,Dum Pukht
255,101349,Surat,North Indian,5455.28,Moti Mahal
1353,100935,Bangalore,Burger,5275.75,Burgerama
488,100119,Kolkata,Biryani,5129.80,Dum Pukht
1220,101145,Mumbai,North Indian,4967.40,Barbeque Nation
802,100877,Chennai,Chinese,4965.35,China Town
123,101126,Hyderabad,Pizza,4893.59,La Pino'z


In [ ]:
# ── Step 6: Before vs After Comparison ────────────────────────────────
print("=" * 50)
print("   BEFORE vs AFTER DATA CLEANING")
print("=" * 50)
print(f"  Rows:             {len(df):>6,} → {len(df_clean):>6,}  ({len(df)-len(df_clean)} removed)")
print(f"  Total GMV:        ₹{df['order_value'].sum():>12,.0f}")
print(f"  Clean GMV:        ₹{df_clean['order_value'].sum():>12,.0f}")
print(f"  AOV (raw):        ₹{df['order_value'].mean():>10.2f}")
print(f"  AOV (clean):      ₹{df_clean['order_value'].mean():>10.2f}")
print(f"  Avg Rating (raw): {pd.to_numeric(df['rating'], errors='coerce').mean():>10.2f}")
print(f"  Avg Rating (cln): {df_clean['rating'].mean():>10.2f}")
print(f"  Nulls (raw):      {df.isnull().sum().sum():>6,} cells")
print(f"  Nulls (clean):    {df_clean.isnull().sum().sum():>6,} cells")
print("=" * 50)

   BEFORE vs AFTER DATA CLEANING
  Rows:              2,050 →  2,000  (50 removed)
  Total GMV:        ₹     910,878
  Clean GMV:        ₹     888,043
  AOV (raw):        ₹    444.33
  AOV (clean):      ₹    444.02
  Avg Rating (raw):       4.00
  Avg Rating (cln):       3.99
  Nulls (raw):       1,348 cells
  Nulls (clean):       896 cells


In [ ]:
# ── Step 7: Export clean dataset ──────────────────────────────────────
df_clean.to_csv('swiggy_orders_clean.csv', index=False)
print("✅ Clean dataset saved as 'swiggy_orders_clean.csv'")
print(f"   Shape: {df_clean.shape[0]:,} rows × {df_clean.shape[1]} columns")
print("\nColumn types in clean dataset:")
print(df_clean.dtypes)

✅ Clean dataset saved as 'swiggy_orders_clean.csv'
   Shape: 2,000 rows × 13 columns

Column types in clean dataset:
order_id                       int64
restaurant_name               object
city                          object
category                      object
order_value                  float64
delivery_time_mins           float64
rating                       float64
order_date            datetime64[ns]
time_slot                     object
is_reorder                      bool
discount_applied             float64
platform                      object
is_outlier                      bool
dtype: object


---
## 📚 Quick Reference: Pandas Cheat Sheet

| Task | Code |
|---|---|
| Load CSV | `pd.read_csv('file.csv', parse_dates=['col'])` |
| Shape | `df.shape` |
| Null audit | `df.isnull().sum()` |
| Filter | `df[df['col'] > value]` |
| Multi-filter | `df[(cond1) & (cond2)]` |
| Sort | `df.sort_values('col', ascending=False)` |
| GroupBy + agg | `df.groupby('col').agg({'val': ['sum', 'mean']})` |
| Pivot table | `pd.pivot_table(df, values, index, columns, aggfunc)` |
| Fill nulls | `df['col'].fillna(df['col'].median())` |
| Remove dupes | `df.drop_duplicates(subset='col', keep='first')` |
| Column to NumPy | `df['col'].values` |
| Vectorised sum | `np.sum(array)` |